In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [187]:
df = pd.read_csv("UpdatedResumeDataSet.csv")
df

,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."
3,Data Science,Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4,Data Science,"Education Details \r\n MCA YMCAUST, Faridab..."
...,...,...
957,Testing,Computer Skills: â¢ Proficient in MS office (...
958,Testing,â Willingness to accept the challenges. â ...
959,Testing,"PERSONAL SKILLS â¢ Quick learner, â¢ Eagerne..."
960,Testing,COMPUTER SKILLS & SOFTWARE KNOWLEDGE MS-Power ...


In [188]:
df['Category'].value_counts()

Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
Sales                        40
Data Science                 40
Mechanical Engineer          40
ETL Developer                40
Blockchain                   40
Operations Manager           40
Arts                         36
Database                     33
Health and fitness           30
PMO                          30
Electrical Engineering       30
Business Analyst             28
DotNet Developer             28
Automation Testing           26
Network Security Engineer    25
Civil Engineer               24
SAP Developer                24
Advocate                     20
Name: count, dtype: int64

## **EDA**

In [189]:
import re

def ResumeCleaner(text):
    cleanText = re.sub('http\S+', ' ', text)  # Remove URLs
    cleanText = re.sub('UTF-?8', ' ', cleanText)  # Remove UTF8 variations
    cleanText = re.sub('\W+', ' ', cleanText)  # Remove non-word characters
    cleanText = re.sub('@\S+', ' ', cleanText)  # Remove mentions
    cleanText = re.sub('[%s]' % re.escape('''!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~'''), ' ', cleanText)  # Remove punctuation
    cleanText = re.sub('[^\x00-\x7F]', ' ', cleanText)  # Remove non-ASCII characters
    cleanText = re.sub('\s+', ' ', cleanText)
    return cleanText

In [190]:
df['Resume'].apply(lambda x: ResumeCleaner(x))

0      Skills Programming Languages Python pandas num...
1      Education Details May 2013 to May 2017 B E UIT...
2      Areas of Interest Deep Learning Control System...
3      Skills R Python SAP HANA Tableau SAP HANA SQL ...
4      Education Details MCA YMCAUST Faridabad Haryan...
                             ...                        
957    Computer Skills Proficient in MS office Word B...
958     Willingness to accept the challenges Positive...
959    PERSONAL SKILLS Quick learner Eagerness to lea...
960    COMPUTER SKILLS SOFTWARE KNOWLEDGE MS Power Po...
961    Skill Set OS Windows XP 7 8 8 1 10 Database MY...
Name: Resume, Length: 962, dtype: object

In [191]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder() 
le.fit(df['Category'])  
df['Category'] = le.transform(df['Category'])  

In [213]:
import joblib
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [171]:
for i, label in enumerate(le.classes_):
    print(f"{i}: {label}")

0: Advocate
1: Arts
2: Automation Testing
3: Blockchain
4: Business Analyst
5: Civil Engineer
6: Data Science
7: Database
8: DevOps Engineer
9: DotNet Developer
10: ETL Developer
11: Electrical Engineering
12: HR
13: Hadoop
14: Health and fitness
15: Java Developer
16: Mechanical Engineer
17: Network Security Engineer
18: Operations Manager
19: PMO
20: Python Developer
21: SAP Developer
22: Sales
23: Testing
24: Web Designing


In [192]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid = TfidfVectorizer(stop_words='english')
tfid.fit(df['Resume'])
newtext = tfid.transform(df['Resume'])

In [173]:
newtext

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 164274 stored elements and shape (962, 7384)>

In [12]:
!pip install xgboost

In [193]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(newtext,df['Category'],random_state=42,train_size=0.2)


## **XG BOOST**

In [32]:
import xgboost
import warnings
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings('ignore')

xgb = OneVsRestClassifier(xgboost.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))
xgb.fit(X_train, y_train)

# Predict probabilities
xgb_y_proba = xgb.predict_proba(X_test)

# Predict class labels
xgb_y_pred = xgb.predict(X_test)




In [33]:
from sklearn.metrics import accuracy_score,roc_auc_score,precision_score,recall_score,f1_score


xgb_acc = accuracy_score(y_test, xgb_y_pred)
xgb_prec = precision_score(y_test, xgb_y_pred, average='macro')
xgb_recall = recall_score(y_test, xgb_y_pred, average='macro')
xgb_f1 = f1_score(y_test, xgb_y_pred, average='macro')
xgb_aucroc = roc_auc_score(y_test, xgb_y_proba, multi_class='ovr')  # or 'ovo'


xgb_results = pd.DataFrame({
    'Accuracy': [xgb_acc],
    'Precision': [xgb_prec],
    'Recall': [xgb_recall],
    'F1 Score': [xgb_f1],
    'AUC ROC': [xgb_aucroc]
})

xgb_results

,Accuracy,Precision,Recall,F1 Score,AUC ROC
0,0.853247,0.836402,0.860236,0.828309,0.992757


## **Decision Tree**

In [38]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings('ignore')

dt = OneVsRestClassifier(DecisionTreeClassifier(criterion='entropy', random_state=42, max_depth=5))
dt.fit(X_train, y_train)

# Predict probabilities
dt_y_proba = dt.predict_proba(X_test)

# Predict class labels
dt_y_pred = dt.predict(X_test)

dt_acc = accuracy_score(y_test, dt_y_pred)
dt_prec = precision_score(y_test, dt_y_pred, average='macro')
dt_recall = recall_score(y_test, dt_y_pred, average='macro')
dt_f1 = f1_score(y_test, dt_y_pred, average='macro')



dt_results = pd.DataFrame({
    'Accuracy': [dt_acc],
    'Precision': [dt_prec],
    'Recall': [dt_recall],
    'F1 Score': [dt_f1],
})

dt_results

,Accuracy,Precision,Recall,F1 Score
0,0.874026,0.928024,0.87227,0.875196


## **KNN**

In [39]:
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')

KNN = OneVsRestClassifier(KNeighborsClassifier())
KNN.fit(X_train, y_train)

# Predict probabilities
KNN_y_proba = KNN.predict_proba(X_test)

# Predict class labels
KNN_y_pred = KNN.predict(X_test)

KNN_acc = accuracy_score(y_test, KNN_y_pred)
KNN_prec = precision_score(y_test, KNN_y_pred, average='macro')
KNN_recall = recall_score(y_test, KNN_y_pred, average='macro')
KNN_f1 = f1_score(y_test, KNN_y_pred, average='macro')



KNN_results = pd.DataFrame({
    'Accuracy': [KNN_acc],
    'Precision': [KNN_prec],
    'Recall': [KNN_recall],
    'F1 Score': [KNN_f1],
})

KNN_results

,Accuracy,Precision,Recall,F1 Score
0,0.757143,0.836653,0.78441,0.76401


## **RandomForest**

In [194]:
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

rf = OneVsRestClassifier(RandomForestClassifier())
rf.fit(X_train, y_train)

# Predict probabilities
rf_y_proba = rf.predict_proba(X_test)

# Predict class labels
rf_y_pred = rf.predict(X_test)

rf_acc = accuracy_score(y_test, rf_y_pred)
rf_prec = precision_score(y_test, rf_y_pred, average='macro')
rf_recall = recall_score(y_test, rf_y_pred, average='macro')
rf_f1 = f1_score(y_test, rf_y_pred, average='macro')



rf_results = pd.DataFrame({
    'Accuracy': [rf_acc],
    'Precision': [rf_prec],
    'Recall': [rf_recall],
    'F1 Score': [rf_f1],
})

rf_results

,Accuracy,Precision,Recall,F1 Score
0,0.828571,0.881505,0.824941,0.814585


## **CatBoost**

In [45]:
!pip install catboost

In [154]:
import catboost

warnings.filterwarnings('ignore')

cb = OneVsRestClassifier(catboost.CatBoostClassifier(verbose=0))
cb.fit(X_train, y_train)

# Predict probabilities
cb_y_proba = cb.predict_proba(X_test)

# Predict class labels
cb_y_pred = cb.predict(X_test)

cb_acc = accuracy_score(y_test, cb_y_pred)
cb_prec = precision_score(y_test, cb_y_pred, average='macro')
cb_recall = recall_score(y_test, cb_y_pred, average='macro')
cb_f1 = f1_score(y_test, cb_y_pred, average='macro')
cb_aucroc = roc_auc_score(y_test, cb_y_proba, multi_class='ovr')



cb_results = pd.DataFrame({
    'Accuracy': [cb_acc],
    'Precision': [cb_prec],
    'Recall': [cb_recall],
    'F1 Score': [cb_f1],
    'AUC ROC' : [cb_aucroc]
})

cb_results

KeyboardInterrupt: 

## **Retarain**

In [175]:
sap = pd.read_csv("SAP_Developer_Projects.csv")


In [159]:
sap['Project_Description'].apply(lambda x: ResumeCleaner(x))

0     Developed custom ABAP reports for inventory tr...
1     Created smart forms and SAP scripts for invoic...
2     Implemented BAPIs and RFCs for integration bet...
3     Enhanced SAP workflow using user exits and BAD...
4     Migrated legacy code to S 4HANA compatible syn...
5     Developed ALV reports to track vendor performa...
6     Built SAP Fiori applications for mobile purcha...
7     Designed CDS views for real time analytics on ...
8     Created OData services to expose SAP data to w...
9     Integrated SAP with Salesforce using SAP PI PO...
10        Customized IDocs for EDI processing in SAP SD
11        Automated invoice posting using BDC in SAP FI
12    Built custom pricing routines in SAP SD using ...
13    Created batch jobs for material master updates...
14    Developed Fiori Elements app using SAPUI5 and ...
15    Performed performance tuning on ABAP reports u...
16    Extended standard SAP reports using implicit e...
17    Configured SAP Gateway for RESTful OData s

In [201]:
mapping_dict = {
    'SAP Developer': 21,
}

# Apply mapping
sap['Category'] = sap['Category'].map(mapping_dict)



In [204]:
bh = pd.read_csv("BusinessAnalyst_Hadoop_Projects.csv")
bh_mapping_dict = {
    'Business Analyst': 4,
    'Hadoop' : 13
}

# Apply mapping
bh['Category'] = bh['Category'].map(bh_mapping_dict)


In [48]:
import pickle
pickle.dump(newtext,open('newtext.pkl','wb'))
pickle.dump(tfid,open('tfid.pkl','wb'))
pickle.dump(cb,open('catboost.pkl','wb'))
pickle.dump(rf,open('randomforest.pkl','wb'))

In [196]:
sap['Category'].unique()

array([21])

## **Prediction**

In [211]:
my_res = 'Designed and developed scalable backend applications using Python, Flask, and Django. Built RESTful APIs, integrated databases with SQLAlchemy and PostgreSQL, and implemented unit testing with PyTest. Automated data processing workflows and deployed applications using Docker and CI/CD pipelines.'

In [212]:
import pickle
cb = pickle.load(open('catboost.pkl','rb'))
res = ResumeCleaner(my_res)

inp_feat = tfid.transform([res])

cb_predict = cb.predict(inp_feat)[0]
dt_predict = dt.predict(inp_feat)[0]

# Create a mapping from label-encoded category to original category name
cat_mapping = {i: label for i, label in enumerate(le.classes_)}

cb_cat_nam = cat_mapping.get(cb_predict,'Unknown')
dt_cat_nam = cat_mapping.get(dt_predict,'Unknown')
print("CatBoost Pridicted Category = ",cb_cat_nam)
print("Decision Tree Pridicted Category = ",dt_cat_nam)


CatBoost Pridicted Category =  Python Developer
Decision Tree Pridicted Category =  Data Science


In [94]:
df['Category'].value_counts()

Category
15    84
23    70
8     55
20    48
24    45
12    44
13    42
22    40
6     40
16    40
10    40
3     40
18    40
1     36
7     33
14    30
19    30
11    30
4     28
9     28
2     26
17    25
5     24
21    24
0     20
Name: count, dtype: int64